# SUB003 -- Multi-Model Ensemble: Template + ResNet + Transformer Refinement

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better, best-of-5 averaged)  
**Lineage**: IT001 → IT002 → IT004 → SUB001 (failed) → SUB002 (0.211) → SUB003

Improvements over SUB002:
- Expanded PDB template database (3000 entries, up from 2000)
- Multi-model refinement ensemble: ResNet + Transformer
- Full use of all experimental structures from validation labels
- Model-diversity for 5-structure generation (maximizes best-of-5 TM-score)
- Confidence-weighted ensemble combining template, ResNet, and Transformer predictions

In [ ]:
import json
import math
import pickle
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

BASE = "/kaggle/input/competitions/stanford-rna-3d-folding-2"
WORK = "/kaggle/working"
DB_DIR = f"{WORK}/template_db"

CFG = {
    # Template DB
    "max_pdb_entries": 3000,
    "max_resolution": 4.5,
    "download_delay": 0.03,
    # Template search
    "top_k_templates": 8,
    "min_identity": 0.15,
    # ResNet refinement
    "resnet_hidden": 128,
    "resnet_layers": 6,
    "resnet_kernel": 5,
    "resnet_dropout": 0.15,
    "resnet_lr": 3e-4,
    "resnet_epochs": 30,
    # Transformer refinement
    "transformer_d_model": 128,
    "transformer_nhead": 4,
    "transformer_layers": 4,
    "transformer_dropout": 0.15,
    "transformer_lr": 3e-4,
    "transformer_epochs": 30,
    # Generation
    "noise_std": 0.3,
    # Ensemble
    "template_weight": 0.4,
    "resnet_weight": 0.3,
    "transformer_weight": 0.3,
}

In [ ]:
def safe_read_csv(path, label=""):
    try:
        df = pd.read_csv(path)
        print(f"Loaded {label}: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"WARNING: {label} not found at {path}")
        return pd.DataFrame()

test_sequences = safe_read_csv(f"{BASE}/test_sequences.csv", "test_sequences")
sample_sub = safe_read_csv(f"{BASE}/sample_submission.csv", "sample_submission")
val_sequences = safe_read_csv(f"{BASE}/validation_sequences.csv", "validation_sequences")
val_labels = safe_read_csv(f"{BASE}/validation_labels.csv", "validation_labels")

print(f"\n--- test_sequences columns: {list(test_sequences.columns)}")
print(f"--- sample_submission columns: {list(sample_sub.columns[:8])} ...")
print(f"--- val_sequences columns: {list(val_sequences.columns)}")
print(f"--- val_labels columns ({len(val_labels.columns)} total): {list(val_labels.columns[:8])} ...")

In [ ]:
# ============================================================
# PDB Template Database
# ============================================================

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_COORD_URL = "https://files.rcsb.org/download"

RNA_RESIDUES = {
    "A", "C", "G", "U", "ADE", "CYT", "GUA", "URA",
    "DA", "DC", "DG", "DT", "RA", "RC", "RG", "RU",
}
THREE_TO_ONE = {
    "A": "A", "ADE": "A", "RA": "A",
    "C": "C", "CYT": "C", "RC": "C",
    "G": "G", "GUA": "G", "RG": "G",
    "U": "U", "URA": "U", "RU": "U",
    "DA": "A", "DC": "C", "DG": "G", "DT": "U",
}


class PDBRNADatabase:
    def __init__(self, db_dir: str = "data/template_db"):
        self.db_dir = Path(db_dir)
        self.db_dir.mkdir(parents=True, exist_ok=True)
        self.index_path = self.db_dir / "template_index.pkl"
        self.index: Dict[str, dict] = {}
        if self.index_path.exists():
            with open(self.index_path, "rb") as f:
                self.index = pickle.load(f)

    def search_rna_entries(self, max_results=5000, min_resolution=0.0, max_resolution=4.0):
        query = {
            "query": {
                "type": "group",
                "logical_operator": "and",
                "nodes": [
                    {
                        "type": "terminal",
                        "service": "text",
                        "parameters": {
                            "attribute": "entity_poly.rcsb_entity_polymer_type",
                            "operator": "exact_match",
                            "value": "RNA",
                        },
                    },
                    {
                        "type": "terminal",
                        "service": "text",
                        "parameters": {
                            "attribute": "rcsb_entry_info.resolution_combined",
                            "operator": "range",
                            "value": {
                                "from": min_resolution,
                                "to": max_resolution,
                                "include_lower": True,
                                "include_upper": True,
                            },
                        },
                    },
                ],
            },
            "return_type": "entry",
            "request_options": {
                "paginate": {"start": 0, "rows": max_results},
                "sort": [{"sort_by": "score", "direction": "desc"}],
            },
        }
        resp = requests.post(RCSB_SEARCH_URL, json=query, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        return [hit["identifier"] for hit in data.get("result_set", [])]

    def download_entry(self, pdb_id: str):
        cache_path = self.db_dir / f"{pdb_id.lower()}.json"
        if cache_path.exists():
            with open(cache_path) as f:
                return json.load(f)
        pdb_url = f"{RCSB_COORD_URL}/{pdb_id.upper()}.pdb"
        try:
            resp = requests.get(pdb_url, timeout=30)
            resp.raise_for_status()
        except Exception:
            return None
        entry = self._parse_pdb_text(pdb_id, resp.text)
        if entry and entry["chains"]:
            with open(cache_path, "w") as f:
                json.dump(entry, f)
        return entry

    def _parse_pdb_text(self, pdb_id, pdb_text):
        chains: Dict[str, dict] = {}
        for line in pdb_text.splitlines():
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue
            atom_name = line[12:16].strip()
            resname = line[17:20].strip()
            chain_id = line[21]
            resid_str = line[22:26].strip()
            if resname not in RNA_RESIDUES:
                continue
            if chain_id not in chains:
                chains[chain_id] = {"residues": {}, "chain_id": chain_id}
            resid = int(resid_str) if resid_str.lstrip("-").isdigit() else 0
            if resid not in chains[chain_id]["residues"]:
                chains[chain_id]["residues"][resid] = {"resname": resname, "atoms": {}}
            try:
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                chains[chain_id]["residues"][resid]["atoms"][atom_name] = [x, y, z]
            except ValueError:
                continue
        result_chains = []
        for cid, cdata in chains.items():
            sorted_resids = sorted(cdata["residues"].keys())
            sequence = ""
            coords = []
            for rid in sorted_resids:
                res = cdata["residues"][rid]
                one_letter = THREE_TO_ONE.get(res["resname"], "N")
                sequence += one_letter
                atom_coord = None
                for preferred in ["C1'", "C3'", "P"]:
                    if preferred in res["atoms"]:
                        atom_coord = res["atoms"][preferred]
                        break
                if atom_coord is None and res["atoms"]:
                    atom_coord = next(iter(res["atoms"].values()))
                if atom_coord is None:
                    atom_coord = [0.0, 0.0, 0.0]
                coords.append(atom_coord)
            if len(sequence) >= 5:
                result_chains.append({"chain_id": cid, "sequence": sequence, "coords": coords})
        return {"pdb_id": pdb_id.upper(), "chains": result_chains}

    def build_database(self, max_entries=2000, max_resolution=4.0, delay=0.1):
        print("Searching RCSB for RNA structures...")
        pdb_ids = self.search_rna_entries(max_results=max_entries, max_resolution=max_resolution)
        print(f"Found {len(pdb_ids)} RNA entries.")
        new_chains = 0
        for pdb_id in tqdm(pdb_ids, desc="Downloading"):
            if pdb_id in self.index:
                continue
            entry = self.download_entry(pdb_id)
            if entry is None:
                continue
            for chain in entry["chains"]:
                key = f"{pdb_id}_{chain['chain_id']}"
                self.index[key] = {
                    "pdb_id": pdb_id,
                    "chain_id": chain["chain_id"],
                    "sequence": chain["sequence"],
                    "length": len(chain["sequence"]),
                    "coords": chain["coords"],
                }
                new_chains += 1
            if delay > 0:
                time.sleep(delay)
        self._save_index()
        print(f"Database built: {len(self.index)} chains total ({new_chains} new).")
        return len(self.index)

    def _save_index(self):
        with open(self.index_path, "wb") as f:
            pickle.dump(self.index, f)

    def search_templates(self, query_sequence, top_k=5, min_identity=0.2):
        query_upper = query_sequence.upper()
        query_kmers = _kmer_set(query_upper, k=4)
        candidates = []
        for key, entry in self.index.items():
            tpl_kmers = _kmer_set(entry["sequence"], k=4)
            if not query_kmers or not tpl_kmers:
                continue
            jaccard = len(query_kmers & tpl_kmers) / len(query_kmers | tpl_kmers)
            if jaccard >= min_identity * 0.3:
                candidates.append((key, entry, jaccard))
        candidates.sort(key=lambda x: -x[2])
        candidates = candidates[: top_k * 5]
        results = []
        for key, entry, _ in candidates:
            identity = sequence_identity(query_upper, entry["sequence"])
            if identity >= min_identity:
                results.append({
                    "key": key,
                    "pdb_id": entry["pdb_id"],
                    "chain_id": entry["chain_id"],
                    "template_sequence": entry["sequence"],
                    "template_length": entry["length"],
                    "identity": identity,
                    "coords": entry["coords"],
                })
        results.sort(key=lambda x: -x["identity"])
        return results[:top_k]


def _kmer_set(seq, k=4):
    return {seq[i : i + k] for i in range(len(seq) - k + 1)} if len(seq) >= k else set()


def sequence_identity(seq_a, seq_b):
    alignment = needleman_wunsch(seq_a, seq_b)
    if alignment["aligned_length"] == 0:
        return 0.0
    return alignment["matches"] / alignment["aligned_length"]


def needleman_wunsch(seq_a, seq_b, match_score=2, mismatch_score=-1, gap_penalty=-2):
    n, m = len(seq_a), len(seq_b)
    if n > 1000 or m > 1000:
        return _banded_nw(seq_a, seq_b, band_width=200, match_score=match_score,
                          mismatch_score=mismatch_score, gap_penalty=gap_penalty)
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    dp[0, :] = np.arange(m + 1) * gap_penalty
    dp[:, 0] = np.arange(n + 1) * gap_penalty
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            dp[i, j] = max(
                dp[i - 1, j - 1] + s,
                dp[i - 1, j] + gap_penalty,
                dp[i, j - 1] + gap_penalty,
            )
    aligned_a, aligned_b = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            if dp[i, j] == dp[i - 1, j - 1] + s:
                aligned_a.append(seq_a[i - 1])
                aligned_b.append(seq_b[j - 1])
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i - 1, j] + gap_penalty:
            aligned_a.append(seq_a[i - 1])
            aligned_b.append("-")
            i -= 1
        else:
            aligned_a.append("-")
            aligned_b.append(seq_b[j - 1])
            j -= 1
    aligned_a.reverse()
    aligned_b.reverse()
    matches = sum(a == b and a != "-" for a, b in zip(aligned_a, aligned_b))
    aligned_length = sum(a != "-" or b != "-" for a, b in zip(aligned_a, aligned_b))
    a_pos, b_pos = -1, -1
    map_a_to_b = {}
    for a_char, b_char in zip(aligned_a, aligned_b):
        if a_char != "-":
            a_pos += 1
        if b_char != "-":
            b_pos += 1
        if a_char != "-" and b_char != "-":
            map_a_to_b[a_pos] = b_pos
    return {
        "aligned_a": "".join(aligned_a),
        "aligned_b": "".join(aligned_b),
        "matches": matches,
        "aligned_length": aligned_length,
        "alignment_map_a_to_b": map_a_to_b,
    }


def _banded_nw(seq_a, seq_b, band_width=200, match_score=2, mismatch_score=-1, gap_penalty=-2):
    """Banded Needleman-Wunsch for long sequences."""
    n, m = len(seq_a), len(seq_b)
    dp = {}
    dp[(0, 0)] = 0
    for i in range(1, min(n + 1, band_width + 1)):
        dp[(i, 0)] = i * gap_penalty
    for j in range(1, min(m + 1, band_width + 1)):
        dp[(0, j)] = j * gap_penalty
    for i in range(1, n + 1):
        j_center = int(i * m / n) if n > 0 else i
        j_start = max(1, j_center - band_width)
        j_end = min(m, j_center + band_width)
        for j in range(j_start, j_end + 1):
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            options = []
            if (i - 1, j - 1) in dp:
                options.append(dp[(i - 1, j - 1)] + s)
            if (i - 1, j) in dp:
                options.append(dp[(i - 1, j)] + gap_penalty)
            if (i, j - 1) in dp:
                options.append(dp[(i, j - 1)] + gap_penalty)
            if options:
                dp[(i, j)] = max(options)
    i, j = n, m
    aligned_a, aligned_b = [], []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and (i - 1, j - 1) in dp:
            s = match_score if seq_a[i - 1] == seq_b[j - 1] else mismatch_score
            if (i, j) in dp and dp[(i, j)] == dp[(i - 1, j - 1)] + s:
                aligned_a.append(seq_a[i - 1])
                aligned_b.append(seq_b[j - 1])
                i -= 1
                j -= 1
                continue
        if i > 0 and (i - 1, j) in dp and (i, j) in dp and dp[(i, j)] == dp[(i - 1, j)] + gap_penalty:
            aligned_a.append(seq_a[i - 1])
            aligned_b.append("-")
            i -= 1
        elif j > 0:
            aligned_a.append("-")
            aligned_b.append(seq_b[j - 1])
            j -= 1
        else:
            break
    aligned_a.reverse()
    aligned_b.reverse()
    matches = sum(a == b and a != "-" for a, b in zip(aligned_a, aligned_b))
    aligned_length = sum(a != "-" or b != "-" for a, b in zip(aligned_a, aligned_b))
    a_pos, b_pos = -1, -1
    map_a_to_b = {}
    for a_char, b_char in zip(aligned_a, aligned_b):
        if a_char != "-":
            a_pos += 1
        if b_char != "-":
            b_pos += 1
        if a_char != "-" and b_char != "-":
            map_a_to_b[a_pos] = b_pos
    return {
        "aligned_a": "".join(aligned_a),
        "aligned_b": "".join(aligned_b),
        "matches": matches,
        "aligned_length": aligned_length,
        "alignment_map_a_to_b": map_a_to_b,
    }


print("Template DB module loaded.")

In [ ]:
# ============================================================
# Template Model + Coordinate Utilities
# ============================================================

def generate_helix_coords(length: int) -> np.ndarray:
    rise_per_residue = 2.81
    radius = 9.4
    twist_per_residue = np.radians(32.7)
    coords = np.zeros((length, 3), dtype=np.float64)
    for i in range(length):
        angle = i * twist_per_residue
        coords[i, 0] = radius * np.cos(angle)
        coords[i, 1] = radius * np.sin(angle)
        coords[i, 2] = i * rise_per_residue
    return coords


def transfer_coordinates(query_sequence, template_sequence, template_coords, alignment_map):
    query_len = len(query_sequence)
    result = np.full((query_len, 3), np.nan, dtype=np.float64)
    for q_pos, t_pos in alignment_map.items():
        if q_pos < query_len and t_pos < len(template_coords):
            result[q_pos] = template_coords[t_pos]
    mapped_positions = sorted(alignment_map.keys())
    if not mapped_positions:
        return generate_helix_coords(query_len)
    for i in range(query_len):
        if not np.isnan(result[i, 0]):
            continue
        left = max((p for p in mapped_positions if p < i), default=None)
        right = min((p for p in mapped_positions if p > i), default=None)
        if left is not None and right is not None:
            frac = (i - left) / (right - left)
            result[i] = result[left] * (1 - frac) + result[right] * frac
        elif left is not None:
            direction = _estimate_direction(result, left, mapped_positions)
            result[i] = result[left] + direction * (i - left)
        elif right is not None:
            direction = _estimate_direction(result, right, mapped_positions)
            result[i] = result[right] - direction * (right - i)
        else:
            result[i] = [0.0, 0.0, 0.0]
    return result


def _estimate_direction(coords, anchor, mapped):
    neighbors = [p for p in mapped if p != anchor and not np.isnan(coords[p, 0])]
    if not neighbors:
        return np.array([0.0, 0.0, 2.81])
    closest = min(neighbors, key=lambda p: abs(p - anchor))
    diff = coords[anchor] - coords[closest]
    dist = np.linalg.norm(diff)
    if dist < 1e-6:
        return np.array([0.0, 0.0, 2.81])
    return diff / max(abs(anchor - closest), 1)


def kabsch_rmsd(P, Q):
    assert P.shape == Q.shape
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    p = P - centroid_P
    q = Q - centroid_Q
    H = q.T @ p
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1.0, 1.0, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    diff = p - q @ R
    rmsd_val = float(np.sqrt((diff**2).sum() / len(P)))
    return rmsd_val, R, centroid_P - centroid_Q @ R


class TemplateModel:
    def __init__(self, database, top_k=5, min_identity=0.2):
        self.database = database
        self.top_k = top_k
        self.min_identity = min_identity

    def predict(self, sequence):
        templates = self.database.search_templates(
            sequence, top_k=self.top_k, min_identity=self.min_identity
        )
        if not templates:
            return {
                "coords": generate_helix_coords(len(sequence)),
                "method": "helix_fallback",
                "templates_used": [],
                "confidence": 0.1,
            }
        predictions = []
        weights = []
        for tpl in templates:
            tpl_coords = np.array(tpl["coords"], dtype=np.float64)
            alignment = needleman_wunsch(sequence.upper(), tpl["template_sequence"])
            transferred = transfer_coordinates(
                sequence.upper(),
                tpl["template_sequence"],
                tpl_coords,
                alignment["alignment_map_a_to_b"],
            )
            predictions.append(transferred)
            weights.append(tpl["identity"])
        w = np.array(weights, dtype=np.float64)
        w = w / w.sum()
        coords = np.zeros_like(predictions[0])
        for pred, wi in zip(predictions, w):
            coords += pred * wi
        return {
            "coords": coords,
            "method": "template",
            "templates_used": [
                {"pdb_id": t["pdb_id"], "chain_id": t["chain_id"], "identity": t["identity"]}
                for t in templates
            ],
            "confidence": float(max(weights)),
            "individual_predictions": predictions,
            "individual_weights": weights,
        }


print("Template model module loaded.")

In [ ]:
# ============================================================
# Build Template Database
# ============================================================

db = PDBRNADatabase(db_dir=DB_DIR)
num_chains = db.build_database(
    max_entries=CFG["max_pdb_entries"],
    max_resolution=CFG["max_resolution"],
    delay=CFG["download_delay"],
)
print(f"\nTemplate database ready: {num_chains} chains indexed.")

In [ ]:
# ============================================================
# Build Training Data from Validation Set
# ============================================================

NT_MAP = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3}

template_model = TemplateModel(
    db, top_k=CFG["top_k_templates"], min_identity=CFG["min_identity"]
)


def detect_structure_columns(label_df):
    """Detect all sets of (x_i, y_i, z_i) columns in validation labels."""
    struct_groups = []
    i = 1
    while True:
        cols = [f"x_{i}", f"y_{i}", f"z_{i}"]
        if all(c in label_df.columns for c in cols):
            struct_groups.append(cols)
            i += 1
        else:
            break
    return struct_groups


def build_training_data(val_seqs_df, val_labels_df):
    """Build paired (template_prediction, ground_truth) from validation data.
    Uses ALL available experimental structures for richer training signal.
    """
    struct_groups = detect_structure_columns(val_labels_df)
    print(f"Detected {len(struct_groups)} structure groups in validation labels.")

    targets = val_seqs_df["target_id"].unique()
    samples = []
    skipped = 0

    for tid in tqdm(targets, desc="Val predictions"):
        seq_row = val_seqs_df[val_seqs_df["target_id"] == tid]
        if seq_row.empty:
            skipped += 1
            continue
        sequence = seq_row.iloc[0]["sequence"]
        if not isinstance(sequence, str) or len(sequence) < 5:
            skipped += 1
            continue

        label_rows = val_labels_df[val_labels_df["ID"].str.startswith(tid + "_")]
        if label_rows.empty:
            skipped += 1
            continue

        pred = template_model.predict(sequence)
        tpl_coords = pred["coords"]
        confidence = pred["confidence"]
        n_res = len(sequence)

        gt_structs = []
        for cols in struct_groups:
            gt = np.full((n_res, 3), np.nan, dtype=np.float64)
            for _, row in label_rows.iterrows():
                parts = row["ID"].split("_")
                resid = int(parts[-1])
                idx = resid - 1
                if 0 <= idx < n_res:
                    vals = row[cols].values.astype(np.float64)
                    if not np.any(np.isnan(vals)):
                        gt[idx] = vals
            if np.isnan(gt).sum() < gt.size * 0.5:
                gt = np.nan_to_num(gt, nan=0.0)
                gt_structs.append(gt)

        if not gt_structs:
            skipped += 1
            continue

        onehot = np.zeros((n_res, 4), dtype=np.float32)
        for i, c in enumerate(sequence.upper()):
            idx = NT_MAP.get(c, -1)
            if idx >= 0:
                onehot[i, idx] = 1.0

        samples.append({
            "target_id": tid,
            "sequence": sequence,
            "seq_onehot": onehot,
            "template_coords": np.array(tpl_coords, dtype=np.float32),
            "gt_coords_list": [g.astype(np.float32) for g in gt_structs],
            "confidence": confidence,
        })

    print(f"Built {len(samples)} training samples ({sum(len(s['gt_coords_list']) for s in samples)} total structures), skipped {skipped}.")
    return samples


if not val_sequences.empty and not val_labels.empty:
    train_samples = build_training_data(val_sequences, val_labels)
else:
    train_samples = []
    print("No validation data available; will skip refinement training.")

In [ ]:
# ============================================================
# Coordinate Normalization Utilities
# ============================================================

def normalize_coords(coords):
    """Center and scale coordinates. Returns (normed, centroid, scale)."""
    centroid = coords.mean(axis=0)
    centered = coords - centroid
    scale = np.std(centered) + 1e-8
    return (centered / scale).astype(np.float32), centroid.astype(np.float32), float(scale)


def denormalize_coords(normed, centroid, scale):
    """Reverse normalize_coords."""
    return normed * scale + centroid


def build_features(template_coords_normed, seq_onehot, confidence):
    """Build (IN_DIM, L) feature tensor from NORMALIZED coords."""
    L = len(seq_onehot)
    rel_pos = np.linspace(0.0, 1.0, L, dtype=np.float32).reshape(-1, 1)
    conf = np.full((L, 1), confidence, dtype=np.float32)
    feats = np.concatenate([
        template_coords_normed.astype(np.float32),
        seq_onehot,
        rel_pos,
        conf,
    ], axis=1)
    return feats  # (L, 9)


FEAT_DIM = 9  # xyz(3) + onehot(4) + rel_pos(1) + confidence(1)
print(f"Feature dimension: {FEAT_DIM}")

In [ ]:
# ============================================================
# Model 1: ResNet Refinement (proven from SUB002)
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dropout, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.InstanceNorm1d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation),
            nn.InstanceNorm1d(channels, affine=True),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.net(x))


class RefinementResNet(nn.Module):
    """1D ResNet predicting coordinate deltas from template features."""

    def __init__(self, in_dim=9, hidden=128, num_blocks=6, kernel_size=5, dropout=0.15):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv1d(in_dim, hidden, 1),
            nn.InstanceNorm1d(hidden, affine=True),
            nn.ReLU(inplace=True),
        )
        blocks = []
        for i in range(num_blocks):
            dilation = 2 ** (i % 4)
            blocks.append(ResidualBlock(hidden, kernel_size, dropout, dilation))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.Conv1d(hidden, hidden // 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv1d(hidden // 2, 3, 1),
        )

    def forward(self, x):
        """x: (B, in_dim, L) -> (B, 3, L) coordinate deltas."""
        h = self.proj(x)
        h = self.blocks(h)
        return self.head(h)


resnet_params = sum(p.numel() for p in RefinementResNet(
    hidden=CFG["resnet_hidden"], num_blocks=CFG["resnet_layers"],
    kernel_size=CFG["resnet_kernel"], dropout=CFG["resnet_dropout"]
).parameters())
print(f"ResNet parameters: {resnet_params:,}")

In [ ]:
# ============================================================
# Model 2: Transformer Refinement (new for SUB003)
# ============================================================

class RefinementTransformer(nn.Module):
    """Transformer encoder predicting coordinate deltas from template features.
    
    Uses pre-norm Transformer with sinusoidal PE and an MLP coordinate head.
    Operates on the same input features as the ResNet refinement model.
    """

    def __init__(self, in_dim=9, d_model=128, nhead=4, num_layers=4,
                 dropout=0.15, max_len=8192):
        super().__init__()
        self.d_model = d_model

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self.coord_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 3),
        )

    def forward(self, x):
        """x: (B, in_dim, L) -> (B, 3, L) coordinate deltas."""
        x = x.transpose(1, 2)  # (B, L, in_dim)
        B, L, _ = x.shape

        h = self.input_proj(x)
        h = h + self.pe[:, :L]
        h = self.encoder(h)
        h = self.final_norm(h)
        delta = self.coord_head(h)  # (B, L, 3)
        return delta.transpose(1, 2)  # (B, 3, L)


transformer_params = sum(p.numel() for p in RefinementTransformer(
    d_model=CFG["transformer_d_model"], nhead=CFG["transformer_nhead"],
    num_layers=CFG["transformer_layers"], dropout=CFG["transformer_dropout"]
).parameters())
print(f"Transformer parameters: {transformer_params:,}")

In [ ]:
# ============================================================
# Training Function (shared for both models)
# ============================================================

def train_refinement_model(model, samples, lr, num_epochs, model_name="Model"):
    """Train a refinement model that predicts coordinate deltas."""
    if not samples:
        print(f"{model_name}: No training samples; returning untrained.")
        return model, False

    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    model.train()
    best_loss = float("inf")
    best_state = None

    for epoch in range(1, num_epochs + 1):
        np.random.shuffle(samples)
        epoch_loss = 0.0
        n_batches = 0

        for sample in samples:
            tpl_c = sample["template_coords"]
            gt_list = sample["gt_coords_list"]
            gt = gt_list[np.random.randint(len(gt_list))]

            tpl_normed, centroid, scale = normalize_coords(tpl_c)
            gt_normed = ((gt - centroid) / scale).astype(np.float32)

            feats = build_features(tpl_normed, sample["seq_onehot"], sample["confidence"])
            feats_t = torch.from_numpy(feats.T).unsqueeze(0).to(DEVICE)  # (1, 9, L)
            tpl_t = torch.from_numpy(tpl_normed.T).unsqueeze(0).to(DEVICE)  # (1, 3, L)
            gt_t = torch.from_numpy(gt_normed.T).unsqueeze(0).to(DEVICE)  # (1, 3, L)

            delta = model(feats_t)
            pred = tpl_t + delta
            loss = ((pred - gt_t) ** 2).mean()

            if not torch.isfinite(loss):
                continue

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f"  {model_name} Epoch {epoch:3d}/{num_epochs}  loss={avg_loss:.6f}  "
                  f"best={best_loss:.6f}  lr={scheduler.get_last_lr()[0]:.2e}")

    training_succeeded = best_loss < 100.0 and np.isfinite(best_loss)
    if best_state is not None and training_succeeded:
        model.load_state_dict(best_state)
    model.eval()
    print(f"{model_name} training complete. Best loss: {best_loss:.6f}  Usable: {training_succeeded}")
    return model, training_succeeded


# Train ResNet
print("=" * 50)
print("Training ResNet Refinement Model")
print("=" * 50)
resnet_model = RefinementResNet(
    hidden=CFG["resnet_hidden"], num_blocks=CFG["resnet_layers"],
    kernel_size=CFG["resnet_kernel"], dropout=CFG["resnet_dropout"]
)
resnet_model, resnet_ok = train_refinement_model(
    resnet_model, train_samples, CFG["resnet_lr"], CFG["resnet_epochs"], "ResNet"
)

# Train Transformer
print("\n" + "=" * 50)
print("Training Transformer Refinement Model")
print("=" * 50)
transformer_model_refine = RefinementTransformer(
    d_model=CFG["transformer_d_model"], nhead=CFG["transformer_nhead"],
    num_layers=CFG["transformer_layers"], dropout=CFG["transformer_dropout"]
)
transformer_model_refine, transformer_ok = train_refinement_model(
    transformer_model_refine, train_samples, CFG["transformer_lr"],
    CFG["transformer_epochs"], "Transformer"
)

print(f"\nResNet usable: {resnet_ok}")
print(f"Transformer usable: {transformer_ok}")

In [ ]:
# ============================================================
# Prediction Functions
# ============================================================

def refine_with_model(model, model_ok, tpl_coords, confidence, onehot,
                      add_noise=0.0, use_dropout=False):
    """Refine template prediction with a trained model."""
    coords = tpl_coords.copy()
    if add_noise > 0:
        coords = coords + np.random.randn(*coords.shape).astype(np.float32) * add_noise

    if not model_ok:
        return coords

    coords_normed, centroid, scale = normalize_coords(coords)
    feats = build_features(coords_normed, onehot, confidence)
    feats_t = torch.from_numpy(feats.T).unsqueeze(0).to(DEVICE)  # (1, 9, L)
    tpl_t = torch.from_numpy(coords_normed.T).unsqueeze(0).to(DEVICE)  # (1, 3, L)

    if use_dropout:
        model.train()
    else:
        model.eval()

    with torch.no_grad():
        delta = model(feats_t)

    refined_normed = (tpl_t + delta).squeeze(0).T.cpu().numpy()
    model.eval()
    return denormalize_coords(refined_normed, centroid, scale)


def generate_5_structures_ensemble(sequence, tpl_coords, confidence, onehot):
    """Generate 5 diverse structures using multi-model ensemble.
    
    Strategy: maximize diversity by using different model combinations
    and noise levels for each of the 5 predictions.
    """
    ns = CFG["noise_std"]
    structures = []

    # Structure 1: Pure template (no refinement noise)
    if resnet_ok and transformer_ok:
        r1 = refine_with_model(resnet_model, True, tpl_coords, confidence, onehot)
        t1 = refine_with_model(transformer_model_refine, True, tpl_coords, confidence, onehot)
        w_r, w_t = CFG["resnet_weight"], CFG["transformer_weight"]
        w_tpl = CFG["template_weight"]
        total = w_r + w_t + w_tpl
        s1 = (r1 * w_r + t1 * w_t + tpl_coords * w_tpl) / total
        structures.append(s1)
    elif resnet_ok:
        structures.append(refine_with_model(resnet_model, True, tpl_coords, confidence, onehot))
    elif transformer_ok:
        structures.append(refine_with_model(transformer_model_refine, True, tpl_coords, confidence, onehot))
    else:
        structures.append(tpl_coords.copy())

    # Structure 2: ResNet with MC-dropout + noise
    structures.append(
        refine_with_model(resnet_model, resnet_ok, tpl_coords, confidence, onehot,
                          add_noise=ns, use_dropout=True)
    )

    # Structure 3: Transformer with MC-dropout + noise
    structures.append(
        refine_with_model(transformer_model_refine, transformer_ok, tpl_coords, confidence, onehot,
                          add_noise=ns, use_dropout=True)
    )

    # Structure 4: ResNet with larger noise
    structures.append(
        refine_with_model(resnet_model, resnet_ok, tpl_coords, confidence, onehot,
                          add_noise=ns * 1.5, use_dropout=True)
    )

    # Structure 5: Transformer with smaller noise (more conservative)
    structures.append(
        refine_with_model(transformer_model_refine, transformer_ok, tpl_coords, confidence, onehot,
                          add_noise=ns * 0.5, use_dropout=False)
    )

    return structures


print("Prediction functions loaded.")

In [ ]:
# ============================================================
# Generate Test Predictions
# ============================================================

if not test_sequences.empty:
    test_targets = test_sequences["target_id"].unique()
    print(f"Generating predictions for {len(test_targets)} test targets...")

    all_predictions = {}
    for tid in tqdm(test_targets, desc="Test predictions"):
        seq_row = test_sequences[test_sequences["target_id"] == tid]
        sequence = seq_row.iloc[0]["sequence"]
        if not isinstance(sequence, str) or len(sequence) == 0:
            sequence = "A"

        pred = template_model.predict(sequence)
        tpl_coords = np.array(pred["coords"], dtype=np.float32)
        confidence = pred["confidence"]

        L = len(sequence)
        onehot = np.zeros((L, 4), dtype=np.float32)
        for i, c in enumerate(sequence.upper()):
            idx = NT_MAP.get(c, -1)
            if idx >= 0:
                onehot[i, idx] = 1.0

        structs = generate_5_structures_ensemble(sequence, tpl_coords, confidence, onehot)
        all_predictions[tid] = structs
    print(f"Generated predictions for {len(all_predictions)} targets.")
else:
    all_predictions = {}
    print("No test sequences found.")

In [ ]:
# ============================================================
# Build Submission CSV
# ============================================================

if not sample_sub.empty and all_predictions:
    coord_col_names = [f"{ax}_{i}" for i in range(1, 6) for ax in ("x", "y", "z")]
    existing_cols = [c for c in sample_sub.columns if c in coord_col_names]
    print(f"Submission shape: {sample_sub.shape}")
    print(f"Coordinate columns found: {len(existing_cols)}")

    for idx, row in tqdm(sample_sub.iterrows(), total=len(sample_sub), desc="Filling submission"):
        parts = row["ID"].rsplit("_", 1)
        if len(parts) != 2:
            continue
        target_id, resid_str = parts
        try:
            resid = int(resid_str)
        except ValueError:
            continue

        if target_id not in all_predictions:
            helix = generate_helix_coords(max(resid, 1))
            vals = []
            for si in range(5):
                ri = min(resid - 1, len(helix) - 1)
                vals.extend(helix[ri].tolist())
            for ci, cn in enumerate(coord_col_names):
                if cn in sample_sub.columns:
                    sample_sub.at[idx, cn] = vals[ci]
            continue

        structs = all_predictions[target_id]
        res_idx = resid - 1

        vals = []
        for si in range(5):
            s = structs[si]
            if res_idx < len(s):
                vals.extend(s[res_idx].tolist())
            else:
                fallback = generate_helix_coords(res_idx + 1)
                vals.extend(fallback[res_idx].tolist())

        for ci, cn in enumerate(coord_col_names):
            if cn in sample_sub.columns:
                sample_sub.at[idx, cn] = vals[ci]

    sample_sub = sample_sub.fillna(0.0)
    output_path = f"{WORK}/submission.csv"
    sample_sub.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")
    print(f"Shape: {sample_sub.shape}")
    print(sample_sub.head())
else:
    print("Cannot generate submission: missing test data or predictions.")
    if sample_sub.empty:
        print("sample_submission.csv was not loaded.")
    if not all_predictions:
        print("No predictions were generated.")

In [ ]:
# ============================================================
# Summary
# ============================================================

print("=" * 60)
print("SUB003 Pipeline Complete")
print("=" * 60)
print(f"  Device used       : {DEVICE}")
print(f"  Template DB size  : {len(db.index)} chains")
print(f"  Training samples  : {len(train_samples)}")
if train_samples:
    total_structs = sum(len(s['gt_coords_list']) for s in train_samples)
    print(f"  Total GT structs  : {total_structs}")
print(f"  ResNet OK         : {resnet_ok}")
print(f"  Transformer OK    : {transformer_ok}")
print(f"  Test targets      : {len(all_predictions)}")
print(f"  Output            : {WORK}/submission.csv")
if not sample_sub.empty:
    print(f"  Submission rows   : {len(sample_sub)}")
    nan_count = sample_sub.isna().sum().sum()
    print(f"  NaN values        : {nan_count}")
print("=" * 60)